# HybridFactor

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam

`HybridFactor` is the abstract base class for all factors involved in hybrid inference problems within GTSAM. It extends the standard `gtsam.Factor` interface to accommodate factors that may involve both discrete and continuous variables.

Key features:
*   Manages separate sets of continuous (`gtsam.Key`) and discrete (`gtsam.DiscreteKey`) variable keys.
*   Provides methods to check the type of factor (discrete-only, continuous-only, or truly hybrid).
*   Defines virtual methods like `errorTree` and `restrict` that derived classes must implement based on their specific nature.

You typically do not instantiate `HybridFactor` directly but work with its derived classes:
*   `gtsam.HybridGaussianFactor`
*   `gtsam.HybridGaussianConditional` (inherits from HybridGaussianFactor)
*   `gtsam.HybridNonlinearFactor`
*   `gtsam.HybridConditional` (wrapper class)

In [ ]:
import gtsam
import numpy as np
from gtsam import DiscreteKey, symbol_shorthand

# Concrete classes for demonstration
from gtsam import HybridGaussianFactor, GaussianFactor, DecisionTreeFactor

X = symbol_shorthand.X
D = symbol_shorthand.D # Example discrete key symbol

## Basic Interface

All hybrid factors provide methods to access their continuous and discrete keys and check their category.

In [ ]:
# Example using a derived class (HybridGaussianFactor)
dk = DiscreteKey(D(0), 2) # Discrete key D0 with cardinality 2
gf0 = gtsam.GaussianFactor(X(0), np.eye(1), np.zeros(1), gtsam.noiseModel.Unit.Create(1))
gf1 = gtsam.GaussianFactor(X(1), np.eye(1), np.ones(1), gtsam.noiseModel.Unit.Create(1))
hybrid_factor = HybridGaussianFactor(dk, [gf0, gf1])

# Access methods inherited from HybridFactor
print(f"Is Discrete? {hybrid_factor.isDiscrete()}")
print(f"Is Continuous? {hybrid_factor.isContinuous()}")
print(f"Is Hybrid? {hybrid_factor.isHybrid()}")
print(f"Number of Continuous Keys: {hybrid_factor.nrContinuous()}")
print(f"Continuous Keys: {hybrid_factor.continuousKeys()}")
print(f"Discrete Keys: {hybrid_factor.discreteKeys()}") # List of (key, card) pairs
print(f"All Keys (Factor base): {hybrid_factor.keys()}") # Combined keys

hybrid_factor.print("\nHybrid Factor Example (HybridGaussianFactor):")

## Virtual Methods (`errorTree`, `restrict`)

Derived classes implement methods to handle the interaction between discrete and continuous parts.

*   `errorTree(VectorValues)`: Calculates the error for the continuous part, returning an `AlgebraicDecisionTree` keyed by the discrete variables.
*   `restrict(DiscreteValues)`: Creates a new factor (typically continuous-only) by fixing the discrete variables to a specific assignment.

In [ ]:
# --- errorTree ---
# Needs continuous values
continuous_values = gtsam.VectorValues()
continuous_values.insert(X(0), np.array([0.5]))
continuous_values.insert(X(1), np.array([1.5]))

try:
    adt = hybrid_factor.errorTree(continuous_values)
    print("\nError Tree (AlgebraicDecisionTree):")
    adt.print()
except Exception as e:
    print(f"\nError calling errorTree: {e}")

# --- restrict ---
# Needs a discrete assignment
assignment = gtsam.DiscreteValues()
assignment[D(0)] = 0 # Choose the first Gaussian factor component

try:
    restricted_factor = hybrid_factor.restrict(assignment)
    # The result is typically a shared_ptr[Factor], often a GaussianFactor
    if restricted_factor:
         restricted_factor.print("\nRestricted Factor (Assignment D0=0):")
         # Cast to GaussianFactor to verify (if possible in python bindings)
         # gaussian_factor = gtsam.GaussianFactor.dynamic_cast(restricted_factor)
         # if gaussian_factor:
         #    print("Successfully cast restricted factor to GaussianFactor")
    else:
        print("\nRestriction resulted in a null factor.")

except Exception as e:
    print(f"\nError calling restrict: {e}")

# Try restricting with the other assignment
assignment[D(0)] = 1
restricted_factor_1 = hybrid_factor.restrict(assignment)
if restricted_factor_1:
    restricted_factor_1.print("\nRestricted Factor (Assignment D0=1):")